#### Faiss
Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [8]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
#from langchain_community.embeddings import OllamaEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader=TextLoader("speech.txt")
documents=loader.load()
text_splitter=CharacterTextSplitter(chunk_size=1000,chunk_overlap=30)
docs=text_splitter.split_documents(documents)


In [9]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct our

In [13]:
embeddings = OllamaEmbeddings(model="gemma:2b")
db = FAISS.from_documents(docs, embeddings)
db

In [14]:
### querying 
query="How does the speaker describe the desired outcome of the war?"
docs=db.similarity_search(query)
docs[0].page_content


'…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [15]:
retriever=db.as_retriever()
docs=retriever.invoke(query)
docs[0].page_content

'…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [16]:
docs_and_score=db.similarity_search_with_score(query)
docs_and_score

[(Document(id='15cdc043-ad3f-45a7-91a5-3ccbbc880e64', metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
  np.float32(0.5535479)),
 (Document(id='c2e0097c-8edb-4c64-b486-470e8cef9bb2', metadata={'source': 'speech.txt'}, page_content='We have borne with their present government through all these bitter months because of that

In [17]:
embedding_vector=embeddings.embed_query(query)
embedding_vector

[0.006769724,
 -0.027601471,
 0.0002012178,
 0.018739449,
 0.008986961,
 0.007331379,
 -0.009418143,
 -0.0015807357,
 -0.0031105643,
 -0.008182903,
 0.017619973,
 -0.0005457346,
 -0.014240807,
 0.010138894,
 0.0037776502,
 -0.010943652,
 0.043859065,
 0.021762189,
 0.017245563,
 0.013113082,
 0.0035680665,
 -0.0057174144,
 0.0032456452,
 0.017845191,
 -0.0006183528,
 -0.008311904,
 -0.017123157,
 -0.019390495,
 -0.00631893,
 -0.022645472,
 -0.0056648087,
 -0.015510251,
 0.018339714,
 -0.013346404,
 -0.0028626106,
 -0.0059189443,
 0.023078727,
 0.006627234,
 0.0010453395,
 -0.004697594,
 0.0043363594,
 0.012096329,
 0.013881264,
 -0.016957447,
 -0.014475225,
 0.004918071,
 0.0024813856,
 0.016535033,
 -0.009533927,
 0.016080651,
 -0.1898221,
 -0.23140785,
 -0.006186832,
 0.004739126,
 0.012632238,
 -0.0046373378,
 -0.019884,
 0.008610235,
 0.006277042,
 -0.005496237,
 0.0028544716,
 -0.016161686,
 0.0005818093,
 -0.015582353,
 -0.0076083047,
 0.0015260811,
 -0.008264273,
 -0.0068513844,

In [18]:
docs_score=db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='15cdc043-ad3f-45a7-91a5-3ccbbc880e64', metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='c2e0097c-8edb-4c64-b486-470e8cef9bb2', metadata={'source': 'speech.txt'}, page_content='We have borne with their present government through all these bitter months because of that friendship—exercising a pat

In [20]:
### Saving And Loading
db.save_local("faiss_index")

In [21]:
new_db=FAISS.load_local("faiss_index",embeddings,allow_dangerous_deserialization=True)
docs=new_db.similarity_search(query)

In [22]:
docs

[Document(id='15cdc043-ad3f-45a7-91a5-3ccbbc880e64', metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='c2e0097c-8edb-4c64-b486-470e8cef9bb2', metadata={'source': 'speech.txt'}, page_content='We have borne with their present government through all these bitter months because of that friendship—exercising a pat